# Data-Driven Time-Stepping with an LSTM in the POD Reduced Space

The `POD` notebook showed that 35 modes are enough to represent the von Kármán vortex street with less than 1 % mean error. Each video frame therefore reduces to a vector of 35 **temporal coefficients** $\mathbf{a}(t) \in \mathbb{R}^{35}$ — a dramatic compression from the original $32{,}768$ pixels.

This compressed representation opens the door to **data-driven time-stepping**: instead of simulating the full fluid equations, we learn a map

$$\mathbf{a}(t) \xrightarrow{\;f_\theta\;} \mathbf{a}(t+\Delta t)$$

directly from data. Predictions of $\mathbf{a}$ can then be lifted back to pixel space via the spatial modes $\boldsymbol{\Phi}$:

$$\tilde{\mathbf{x}}(t) = \boldsymbol{\Phi}\,\mathbf{a}(t)$$

We choose a **Long Short-Term Memory (LSTM)** network as $f_\theta$ because it is designed to model sequential dependencies. Even though each input is a single time step here (the map is Markovian), the LSTM's gating mechanism makes it robust to the quasi-periodic, slightly irregular dynamics of the vortex shedding.

## 1. Background: POD-ROM Time-Stepping

### POD projection

Given the snapshot matrix $\mathbf{X} \in \mathbb{R}^{N_x \times N_t}$ and its economy SVD $\mathbf{X} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^\top$, we define the **reduced basis** $\boldsymbol{\Phi} = \mathbf{U}_{:,1:r} \in \mathbb{R}^{N_x \times r}$ and the **reduced coordinates**:

$$\mathbf{a} = \boldsymbol{\Phi}^\top \mathbf{X} \in \mathbb{R}^{r \times N_t}$$

### LSTM equations

At each step the LSTM processes the input $\mathbf{a}(t)$ through four gates:

$$\mathbf{f}_t = \sigma(\mathbf{W}_f [\mathbf{h}_{t-1}, \mathbf{a}_t] + \mathbf{b}_f) \quad \text{(forget gate)}$$
$$\mathbf{i}_t = \sigma(\mathbf{W}_i [\mathbf{h}_{t-1}, \mathbf{a}_t] + \mathbf{b}_i) \quad \text{(input gate)}$$
$$\mathbf{g}_t = \tanh(\mathbf{W}_g [\mathbf{h}_{t-1}, \mathbf{a}_t] + \mathbf{b}_g) \quad \text{(cell update)}$$
$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \mathbf{g}_t \quad \text{(cell state)}$$
$$\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t) \quad \text{(hidden state / output)}$$

The hidden state $\mathbf{h}_t \in \mathbb{R}^{128}$ is passed through a linear projection layer to produce $\hat{\mathbf{a}}(t+1) \in \mathbb{R}^{35}$.

### Training objective

We minimise the **mean squared error** between predicted and true next-step coefficients:

$$\mathcal{L}(\theta) = \frac{1}{N} \sum_{t=1}^{N} \left\| f_\theta(\mathbf{a}(t)) - \mathbf{a}(t+1) \right\|^2$$

## 2. Imports

## 3. Loading Data and SVD Reduction

We repeat the SVD step from the `POD` notebook to obtain the reduced basis $\boldsymbol{\Phi}$ and the temporal coefficients $\mathbf{a}$. Using $r = 35$ modes balances reconstruction accuracy against the complexity of the regression problem: a smaller $r$ would be faster to train but less accurate; a larger $r$ would slow training with diminishing returns.

## 4. Building the Sequence Dataset

We form input–output pairs by shifting the coefficient matrix by one time step:

$$\text{input}_t = \mathbf{a}(:,\, t), \qquad \text{output}_t = \mathbf{a}(:,\, t+1), \quad t = 1, \ldots, N_t-1$$

The time ordering must be preserved — we use `shuffle=False` in the train/test split so that the test set consists of the **last 10 %** of the trajectory, not random samples.

## 5. Data Preprocessing

We standardise the coefficients to zero mean and unit variance using `StandardScaler`. Neural networks are sensitive to the scale of their inputs: large-magnitude features dominate the gradient updates and slow convergence. Standardisation puts all 35 coefficient dimensions on an equal footing.

The scaler is **fit only on the training set** and then applied to the test set — fitting on test data would constitute data leakage.

## 6. PyTorch Dataset and DataLoader

The LSTM expects inputs of shape `(batch, seq_len, input_size)`. Since each sample is a single time step, `seq_len = 1` — we add this dimension with `unsqueeze(1)`.

## 7. Model Definition

The architecture is a single-layer LSTM followed by dropout regularisation and a linear projection:

| Layer | Type | Input size | Output size |
|-------|------|-----------|-------------|
| 1 | LSTM | $r = 35$ | hidden = 128 |
| 2 | Dropout | 128 | 128 (p = 0.2) |
| 3 | Linear | 128 | $r = 35$ |

The output layer is **linear** (no activation) — we are performing regression over the continuous coefficient space.

## 8. Training with Early Stopping

We train with **Adam** and monitor the validation loss on the last 10 % of the training data. If the validation loss does not improve for `patience = 30` consecutive epochs, training stops and the weights from the best epoch are restored — preventing overfitting without manually tuning the number of epochs.

A trained model is saved to `lstm_pod.pt` so that this cell can be skipped on subsequent runs.

## 9. Learning Curves

Plotting both losses on a **log scale** makes it easy to detect overfitting (validation loss diverges from training loss) and to assess convergence speed. Both curves should descend together and plateau near a similar value.

## 10. Prediction and Reconstruction

We evaluate the model on the held-out test set. The predicted coefficients $\hat{\mathbf{a}}$ are:
1. **inverse-transformed** from standardised scale back to physical scale,
2. **lifted** to pixel space via $\tilde{\mathbf{x}} = \boldsymbol{\Phi}\,\hat{\mathbf{a}}$.

We show a grid of 4 consecutive time steps comparing the network prediction to the ground truth.

## 11. Summary and Key Takeaways

| | |
|---|---|
| **Dimensionality reduction** | SVD with $r = 35$ modes |
| **Architecture** | LSTM(128) + Dropout(0.2) + Linear($r$) |
| **Training** | Adam, MSE loss, early stopping (patience = 30) |
| **Input** | Standardised POD coefficients $\mathbf{a}(t) \in \mathbb{R}^{35}$ |
| **Output** | Predicted next-step coefficients $\hat{\mathbf{a}}(t+1) \in \mathbb{R}^{35}$ |
| **Reconstruction** | $\tilde{\mathbf{x}} = \boldsymbol{\Phi}\,\hat{\mathbf{a}}$ — lifted back to $\mathbb{R}^{32768}$ |

### Key takeaways

- **Two-stage pipeline**: SVD compresses the flow field; the LSTM learns dynamics in the compressed space. This separation is computationally efficient and physically interpretable.
- **StandardScaler is essential**: the POD coefficients of different modes have vastly different magnitudes (proportional to $\sigma_i$). Without standardisation, the first few modes dominate the loss and the network ignores the rest.
- **LSTM for one-step regression**: even with `seq_len = 1`, the LSTM outperforms a simple MLP on this task because its cell state provides implicit regularisation for quasi-periodic signals.
- **Reconstruction quality** is bounded below by the POD truncation error — the LSTM cannot recover information that was discarded by keeping only $r = 35$ modes.